In [1]:
import os
import requests
from dotenv import load_dotenv

import google.generativeai as genai

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings


# ============================
# Load Environment Variables
# ============================

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
JINA_API_KEY = os.getenv("JINA_API_KEY")

if GOOGLE_API_KEY is None:
    raise ValueError("GOOGLE_API_KEY not found.")

if JINA_API_KEY is None:
    raise ValueError("JINA_API_KEY not found.")


# ============================
# Configure Gemini
# ============================

genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel(
    "gemini-2.5-flash"
)


# ============================
# Jina Embedding Class
# ============================

class JinaEmbedding(Embeddings):

    def __init__(self, api_key):
        self.api_key = api_key
        self.url = "https://api.jina.ai/v1/embeddings"

    def embed_documents(self, texts):

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        embeddings = []

        print("Creating embeddings...")

        for i, text in enumerate(texts):

            payload = {
                "model": "jina-embeddings-v3",
                "input": [text]
            }

            response = requests.post(
                self.url,
                headers=headers,
                json=payload
            )

            response.raise_for_status()

            embedding = response.json()["data"][0]["embedding"]

            embeddings.append(embedding)

            print(f"Embedded chunk {i+1}/{len(texts)}")

        return embeddings

    def embed_query(self, text):

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": "jina-embeddings-v3",
            "input": [text]
        }

        response = requests.post(
            self.url,
            headers=headers,
            json=payload
        )

        response.raise_for_status()

        return response.json()["data"][0]["embedding"]


# ============================
# Load PDF
# ============================

print("Loading PDF...")

loader = PyPDFLoader("insurance_policy.pdf")

documents = loader.load()

print(f"Pages Loaded : {len(documents)}")


# ============================
# Split PDF
# ============================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print(f"Chunks Created : {len(chunks)}")


# ============================
# Create Embeddings
# ============================

embeddings = JinaEmbedding(JINA_API_KEY)


# ============================
# Create Vector Database
# ============================

print("Creating FAISS Vector Database...")

vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("Vector Database Created Successfully")


# ============================
# Retrieval Function
# ============================

def retrieve(question):

    docs = vector_db.similarity_search(
        question,
        k=3
    )

    return docs


# ============================
# Ask Questions
# ============================

while True:

    question = input("\nAsk a Question (type exit to quit): ")

    if question.lower() == "exit":
        break

    docs = retrieve(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
You are an Automobile Insurance Policy Expert.

Answer ONLY using the context below.

If the answer is not found in the context, say:

"I could not find the answer in the insurance policy."

Context:
{context}

Question:
{question}

Answer:
"""

    response = model.generate_content(prompt)

    print("\nAnswer:\n")
    print(response.text)

C:\Anaconda3\envs\genai\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Anaconda3\envs\genai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\AppData\Local\Temp\ipykernel_4804\4212417799.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/googl

Loading PDF...
Pages Loaded : 21
Chunks Created : 347
Creating FAISS Vector Database...
Creating embeddings...
Embedded chunk 1/347
Embedded chunk 2/347
Embedded chunk 3/347
Embedded chunk 4/347
Embedded chunk 5/347
Embedded chunk 6/347
Embedded chunk 7/347
Embedded chunk 8/347
Embedded chunk 9/347
Embedded chunk 10/347
Embedded chunk 11/347
Embedded chunk 12/347
Embedded chunk 13/347
Embedded chunk 14/347
Embedded chunk 15/347
Embedded chunk 16/347
Embedded chunk 17/347
Embedded chunk 18/347
Embedded chunk 19/347
Embedded chunk 20/347
Embedded chunk 21/347
Embedded chunk 22/347
Embedded chunk 23/347
Embedded chunk 24/347
Embedded chunk 25/347
Embedded chunk 26/347
Embedded chunk 27/347
Embedded chunk 28/347
Embedded chunk 29/347
Embedded chunk 30/347
Embedded chunk 31/347
Embedded chunk 32/347
Embedded chunk 33/347
Embedded chunk 34/347
Embedded chunk 35/347
Embedded chunk 36/347
Embedded chunk 37/347
Embedded chunk 38/347
Embedded chunk 39/347
Embedded chunk 40/347
Embedded chunk 41/


Ask a Question (type exit to quit):  what is current insurance basics?



Answer:

I could not find the answer in the insurance policy.



Ask a Question (type exit to quit):  purpose of insurance



Answer:

The purpose of the insurance is in respect of accidental loss or damage occurring during the period of insurance. The Company will indemnify the Insured in the event of an accident caused by or arising out of the use of the vehicle against all sums which the Insured shall become legally liable to pay in respect of death of or bodily injury to any person including occupants carried in the vehicle.



Ask a Question (type exit to quit):  exit


In [2]:
import os

print("Current Folder:")
print(os.getcwd())

print("\nFiles inside rag_db:")
print(os.listdir("rag_db"))

Current Folder:
E:\HOPE\AI CLASS\gen ai\INSURANCE_CLAIM_AI_AGENT\notebooks

Files inside rag_db:


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'rag_db'